In [12]:
import pandas as pd
import numpy as np

In [13]:
import pandas as pd
import numpy as np

train = pd.read_parquet(
    "../data/processed/train_personalized.parquet"
)

validation = pd.read_parquet(
    "../data/processed/validation_personalized.parquet"
)

test = pd.read_parquet(
    "../data/processed/test_personalized.parquet"
)

movies = pd.read_csv(
    "/Users/abedhtiwari/Self-Project/personalized-content-ranking/data/raw/ml-32m/movies.csv"
)

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)
print("Movies:", movies.shape)

Train: (22488423, 7)
Validation: (4803475, 7)
Test: (4708306, 7)
Movies: (87585, 3)


In [14]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (22488423, 7)
Validation: (4803475, 7)
Test: (4708306, 7)


In [15]:
movie_popularity = (
    train.groupby("movieId")
    .size()
    .sort_values(ascending=False)
)

In [16]:
top_movies = movie_popularity.head(20)

top_movies

movieId
318     91335
356     84986
296     84620
2571    82265
593     77600
260     73013
2959    65940
4993    62745
527     62462
480     62159
1196    61642
7153    58344
110     58198
5952    57863
1198    57296
1       56844
1210    56631
589     56424
50      55728
858     55613
dtype: int64

In [17]:
top_movies_df = (
    top_movies
    .rename("positive_interactions")
    .reset_index()
    .merge(movies, on="movieId", how="left")
)

In [18]:
top_movies_df[
    ["movieId", "title", "genres", "positive_interactions"]
]

,movieId,title,genres,positive_interactions
0,318,"Shawshank Redemption, The (1994)",Crime|Drama,91335
1,356,Forrest Gump (1994),Comedy|Drama|Romance|War,84986
2,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,84620
3,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,82265
4,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,77600
5,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,73013
6,2959,Fight Club (1999),Action|Crime|Drama|Thriller,65940
7,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,62745
8,527,Schindler's List (1993),Drama|War,62462
9,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,62159


In [19]:
top_10_popular = (
    movie_popularity
    .head(10)
    .index
    .tolist()
)

In [20]:
top_10_popular

[318, 356, 296, 2571, 593, 260, 2959, 4993, 527, 480]

In [22]:
for k in [10, 20, 50, 100]:
    score = popularity_recall_at_k(
        movie_popularity,
        test,
        k
    )
    
    print(f"Recall@{k}: {score:.4f}")

NameError: name 'popularity_recall_at_k' is not defined

In [ ]:
from scipy.sparse import csr_matrix

In [23]:
user_ids = train["userId"].unique()
movie_ids = train["movieId"].unique()

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(movie_ids)
}

In [24]:
rows = train["userId"].map(user_to_idx)
cols = train["movieId"].map(movie_to_idx)

data = np.ones(len(train))

user_item_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(user_ids), len(movie_ids))
)

NameError: name 'csr_matrix' is not defined

In [25]:
print(user_item_matrix.shape)

NameError: name 'user_item_matrix' is not defined

In [26]:
print(user_item_matrix.shape)
print(user_item_matrix.nnz) 
# nnz - number of non-zero entries

NameError: name 'user_item_matrix' is not defined

In [27]:
from sklearn.neighbors import NearestNeighbors

In [28]:
item_user_matrix = user_item_matrix.T.tocsr()

print(item_user_matrix.shape)

NameError: name 'user_item_matrix' is not defined

In [29]:
item_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=51,
    n_jobs=-1
)

In [30]:
item_model.fit(item_user_matrix)

NameError: name 'item_user_matrix' is not defined

In [31]:
sample_movie_id = train["movieId"].iloc[0]

sample_movie_id

np.int64(1176)

In [32]:
sample_movie_idx = movie_to_idx[sample_movie_id]

In [33]:
distances, indices = item_model.kneighbors(
    item_user_matrix[sample_movie_idx],
    n_neighbors=11
)

NameError: name 'item_user_matrix' is not defined

In [34]:
print("Movie ID:", sample_movie_id)
print("Neighbor indices:", indices[0])
print("Distances:", distances[0])

Movie ID: 1176


NameError: name 'indices' is not defined

In [35]:
similarities = 1 - distances[0]
similarities

NameError: name 'distances' is not defined

In [36]:
neighbor_movie_ids = [
    movie_ids[idx]
    for idx in indices[0]
]

NameError: name 'indices' is not defined

In [37]:
similar_movies = (
    movies[movies["movieId"].isin(neighbor_movie_ids)]
    .copy()
)

NameError: name 'neighbor_movie_ids' is not defined

In [38]:
similar_movies[
    ["movieId", "title", "genres"]
]

NameError: name 'similar_movies' is not defined

In [39]:
similar_movies = pd.DataFrame({
    "movieId": neighbor_movie_ids,
    "similarity": similarities
}).merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

NameError: name 'neighbor_movie_ids' is not defined

In [40]:
similar_movies

NameError: name 'similar_movies' is not defined

In [41]:
def recommend_for_user(user_id, k=10, n_neighbors=11):
    
    # Get movies this user positively interacted with
    user_history = train[
        train["userId"] == user_id
    ]["movieId"].tolist()
    
    # Keep only movies that exist in our training index
    user_history = [
        movie_id
        for movie_id in user_history
        if movie_id in movie_to_idx
    ]
    
    # Convert movie IDs to matrix indices
    history_indices = [
        movie_to_idx[movie_id]
        for movie_id in user_history
    ]
    
    # Get all movie vectors for this user's history
    history_matrix = item_user_matrix[history_indices]
    
    # Find neighbors for ALL history movies in one call
    distances, indices = item_model.kneighbors(
        history_matrix,
        n_neighbors=n_neighbors
    )
    
    # Aggregate recommendation scores
    recommendation_scores = {}
    
    for row in range(len(user_history)):
        
        similarities = 1 - distances[row]
        
        for idx, similarity in zip(
            indices[row],
            similarities
        ):
            
            candidate_movie_id = movie_ids[idx]
            
            # Don't recommend movies the user already interacted with
            if candidate_movie_id in user_history:
                continue
            
            recommendation_scores[candidate_movie_id] = (
                recommendation_scores.get(candidate_movie_id, 0)
                + similarity
            )
    
    # Sort by aggregated score
    recommendations = sorted(
        recommendation_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )
    
    return recommendations[:k]

In [42]:
sample_user = (
    train.groupby("userId")
    .size()
    .sort_values(ascending=False)
    .index[1000]
)

sample_user

np.int64(111765)

In [43]:
user_history = train[
    train["userId"] == sample_user
][["movieId", "rating", "date"]].sort_values("date")

user_history.head(20)

,movieId,rating,date
9789548,1957,4.0,2007-08-26 22:13:34
9789549,1104,5.0,2007-08-26 22:14:06
9789552,1230,5.0,2007-08-26 22:25:16
9789553,909,4.5,2007-08-26 22:25:21
9789554,110,3.0,2007-08-26 22:26:08
9789555,912,5.0,2007-08-26 22:26:33
9789556,1183,4.0,2007-08-26 22:26:52
9789557,1953,4.0,2007-08-26 22:27:02
9789558,858,5.0,2007-08-26 22:27:19
9789559,1221,5.0,2007-08-26 22:27:21


In [44]:
recommendations = recommend_for_user(
    sample_user,
    k=10
)

recommendations

NameError: name 'item_user_matrix' is not defined

In [45]:
recommendation_df = pd.DataFrame(
    recommendations,
    columns=["movieId", "cf_score"]
)

NameError: name 'recommendations' is not defined

In [46]:
recommendation_df = recommendation_df.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

NameError: name 'recommendation_df' is not defined

In [47]:
recommendation_df

NameError: name 'recommendation_df' is not defined

In [48]:
user_history

,movieId,rating,date
9789548,1957,4.0,2007-08-26 22:13:34
9789549,1104,5.0,2007-08-26 22:14:06
9789552,1230,5.0,2007-08-26 22:25:16
9789553,909,4.5,2007-08-26 22:25:21
9789554,110,3.0,2007-08-26 22:26:08
...,...,...,...
21170190,1916,3.5,2021-06-01 21:14:20
21170191,2745,3.5,2021-06-01 21:15:47
21170192,2916,3.5,2021-06-01 21:15:55
21170193,3148,3.5,2021-06-01 21:15:58


In [49]:
recommended_ids = {
    movie_id
    for movie_id, score in recommendations
}

seen_ids = set(
    train[train["userId"] == sample_user]["movieId"]
)

print(
    "Recommended movies already seen:",
    len(recommended_ids & seen_ids)
)

NameError: name 'recommendations' is not defined

In [50]:
sample_user = (
    train.groupby("userId")
    .size()
    .sort_values(ascending=False)
    .index[1000]
)

In [51]:
user_history = train[
    train["userId"] == sample_user
][["movieId", "rating", "date"]].sort_values("date")

user_history.head(20)

,movieId,rating,date
9789548,1957,4.0,2007-08-26 22:13:34
9789549,1104,5.0,2007-08-26 22:14:06
9789552,1230,5.0,2007-08-26 22:25:16
9789553,909,4.5,2007-08-26 22:25:21
9789554,110,3.0,2007-08-26 22:26:08
9789555,912,5.0,2007-08-26 22:26:33
9789556,1183,4.0,2007-08-26 22:26:52
9789557,1953,4.0,2007-08-26 22:27:02
9789558,858,5.0,2007-08-26 22:27:19
9789559,1221,5.0,2007-08-26 22:27:21


In [52]:
recommendations = recommend_for_user(
    sample_user,
    k=10
)

recommendations

NameError: name 'item_user_matrix' is not defined

In [53]:
recommendation_df

NameError: name 'recommendation_df' is not defined

In [54]:
print(
    "Recommended movies already seen:",
    len(recommended_ids & seen_ids)
)

NameError: name 'recommended_ids' is not defined

In [55]:
print("Sample user:", sample_user)

print(
    "Number of positive interactions:",
    len(user_history)
)

Sample user: 111765
Number of positive interactions: 1175


In [56]:
print(user_history)

          movieId  rating                date
9789548      1957     4.0 2007-08-26 22:13:34
9789549      1104     5.0 2007-08-26 22:14:06
9789552      1230     5.0 2007-08-26 22:25:16
9789553       909     4.5 2007-08-26 22:25:21
9789554       110     3.0 2007-08-26 22:26:08
...           ...     ...                 ...
21170190     1916     3.5 2021-06-01 21:14:20
21170191     2745     3.5 2021-06-01 21:15:47
21170192     2916     3.5 2021-06-01 21:15:55
21170193     3148     3.5 2021-06-01 21:15:58
21170194     6666     3.5 2021-06-01 21:17:37

[1175 rows x 3 columns]


In [57]:
all_distances, all_indices = item_model.kneighbors(
    item_user_matrix,
    n_neighbors=51
)

all_similarities = 1 - all_distances

print("Neighbor matrix:", all_indices.shape)
print("Similarity matrix:", all_similarities.shape)

NameError: name 'item_user_matrix' is not defined

In [58]:
def recommend_for_user_fast(user_id, k=10):
    
    # Get movies this user positively interacted with
    user_history = train[
        train["userId"] == user_id
    ]["movieId"].tolist()
    
    # Keep only movies available in the training index
    user_history = [
        movie_id
        for movie_id in user_history
        if movie_id in movie_to_idx
    ]
    
    seen_movies = set(user_history)
    
    recommendation_scores = {}
    
    # Use precomputed neighbors
    for movie_id in user_history:
        
        movie_idx = movie_to_idx[movie_id]
        
        neighbor_indices = all_indices[movie_idx]
        similarities = all_similarities[movie_idx]
        
        for idx, similarity in zip(
            neighbor_indices,
            similarities
        ):
            
            candidate_movie_id = movie_ids[idx]
            
            # Don't recommend already-seen movies
            if candidate_movie_id in seen_movies:
                continue
            
            recommendation_scores[candidate_movie_id] = (
                recommendation_scores.get(candidate_movie_id, 0)
                + similarity
            )
    
    recommendations = sorted(
        recommendation_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )
    
    return recommendations[:k]

In [59]:
import time

start = time.time()

fast_recommendations = recommend_for_user_fast(
    sample_user,
    k=10
)

end = time.time()

print(f"Runtime: {end - start:.4f} seconds")
print(fast_recommendations)

NameError: name 'all_indices' is not defined

In [60]:
fast_recommendation_df = pd.DataFrame(
    fast_recommendations,
    columns=["movieId", "cf_score"]
)

fast_recommendation_df = fast_recommendation_df.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

fast_recommendation_df

NameError: name 'fast_recommendations' is not defined

In [61]:
recommended_ids = {
    movie_id
    for movie_id, score in fast_recommendations
}

seen_movies = set(
    train[
        train["userId"] == sample_user
    ]["movieId"]
)

print(
    "Recommended movies already seen:",
    len(recommended_ids & seen_movies)
)

NameError: name 'fast_recommendations' is not defined

In [62]:
fast_recommendations = recommend_for_user_fast(
    sample_user,
    k=10
)

NameError: name 'all_indices' is not defined

In [63]:
np.save(
    "../data/processed/item_neighbor_indices.npy",
    all_indices
)

np.save(
    "../data/processed/item_neighbor_similarities.npy",
    all_similarities
)

NameError: name 'all_indices' is not defined

In [64]:
print("Saved precomputed item neighbors.")

Saved precomputed item neighbors.


In [65]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)
print("Movies:", movies.shape)

Train: (22488423, 7)
Validation: (4803475, 7)
Test: (4708306, 7)
Movies: (87585, 3)


In [66]:
test_positive_by_user = (
    test[test["positive"] == 1]
    .groupby("userId")["movieId"]
    .apply(set)
)

train_seen_by_user = (
    train
    .groupby("userId")["movieId"]
    .apply(set)
)

In [67]:
# ============================================
# EVALUATION METRICS
# ============================================

def recall_at_k(recommended, relevant, k):
    recommended = recommended[:k]
    
    if len(relevant) == 0:
        return 0.0
    
    hits = len(set(recommended) & set(relevant))
    
    return hits / len(relevant)


def precision_at_k(recommended, relevant, k):
    recommended = recommended[:k]
    
    if k == 0:
        return 0.0
    
    hits = len(set(recommended) & set(relevant))
    
    return hits / k


def ndcg_at_k(recommended, relevant, k):
    recommended = recommended[:k]
    relevant = set(relevant)
    
    if len(relevant) == 0:
        return 0.0
    
    dcg = 0.0
    
    for rank, movie_id in enumerate(recommended, start=1):
        if movie_id in relevant:
            dcg += 1 / np.log2(rank + 1)
    
    ideal_hits = min(len(relevant), k)
    
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )
    
    return dcg / idcg if idcg > 0 else 0.0

In [68]:
# ============================================
# BASELINE 1 — POPULARITY
# ============================================

movie_positive_counts = (
    train[train["positive"] == 1]
    .groupby("movieId")
    .size()
    .sort_values(ascending=False)
)

print("Number of movies with positive interactions:",
      len(movie_positive_counts))

print("\nTop 10 popular movies:")
display(movie_positive_counts.head(10))

Number of movies with positive interactions: 43417

Top 10 popular movies:


movieId
318     78889
296     66862
2571    63375
356     61723
593     60932
260     55187
2959    52852
527     50134
1196    47498
4993    47140
dtype: int64

In [69]:
# ============================================
# POPULARITY BASELINE — TOP MOVIES
# Add movie titles and genres
# ============================================

top_popular = (
    movie_positive_counts
    .head(10)
    .reset_index(name="positive_interactions")
    .merge(
        movies,
        on="movieId",
        how="left"
    )
)

display(top_popular)

,movieId,positive_interactions,title,genres
0,318,78889,"Shawshank Redemption, The (1994)",Crime|Drama
1,296,66862,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
2,2571,63375,"Matrix, The (1999)",Action|Sci-Fi|Thriller
3,356,61723,Forrest Gump (1994),Comedy|Drama|Romance|War
4,593,60932,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
5,260,55187,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
6,2959,52852,Fight Club (1999),Action|Crime|Drama|Thriller
7,527,50134,Schindler's List (1993),Drama|War
8,1196,47498,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi
9,4993,47140,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy


In [70]:
# Top popular movies based on positive interactions in training
popular_movies = movie_positive_counts.index.tolist()

print("Number of popular movies:", len(popular_movies))
print("Top 10:", popular_movies[:10])

Number of popular movies: 43417
Top 10: [318, 296, 2571, 356, 593, 260, 2959, 527, 1196, 4993]


In [71]:
# Evaluate the popularity baseline at K=10

K = 10

recalls = []
precisions = []
ndcgs = []

for user_id, relevant_movies in test_positive_by_user.items():

    # Movies this user has already seen during training
    seen_movies = train_seen_by_user[user_id]

    # Recommend the most popular unseen movies
    recommendations = [
        movie_id
        for movie_id in popular_movies
        if movie_id not in seen_movies
    ][:K]

    # Evaluate
    recalls.append(
        recall_at_k(recommendations, relevant_movies, K)
    )

    precisions.append(
        precision_at_k(recommendations, relevant_movies, K)
    )

    ndcgs.append(
        ndcg_at_k(recommendations, relevant_movies, K)
    )

print(f"Users evaluated: {len(recalls):,}")
print(f"Popularity Recall@{K}:    {np.mean(recalls):.4f}")
print(f"Popularity Precision@{K}: {np.mean(precisions):.4f}")
print(f"Popularity NDCG@{K}:      {np.mean(ndcgs):.4f}")

Users evaluated: 193,622
Popularity Recall@10:    0.0480
Popularity Precision@10: 0.0382
Popularity NDCG@10:      0.0530


In [72]:
# Check whether the top 100 popular movies
# are enough to provide 10 unseen recommendations per user

candidate_pool = popular_movies[:100]

users_with_less_than_10 = 0

for user_id in test_positive_by_user.index:

    # Convert the user's training history to a set
    seen_movies = set(train_seen_by_user.loc[user_id])

    unseen_count = sum(
        movie_id not in seen_movies
        for movie_id in candidate_pool
    )

    if unseen_count < 10:
        users_with_less_than_10 += 1

print("Users evaluated:", len(test_positive_by_user))
print(
    "Users with fewer than 10 unseen movies in top 100:",
    users_with_less_than_10
)

Users evaluated: 193622
Users with fewer than 10 unseen movies in top 100: 417


In [73]:
K = 10

popular_array = np.array(popular_movies)

recalls = []
precisions = []
ndcgs = []

for user_id, relevant_movies in test_positive_by_user.items():

    seen_movies = train_seen_by_user.loc[user_id]

    # Remove movies already seen during training
    unseen_popular = popular_array[
        ~np.isin(popular_array, list(seen_movies))
    ]

    recommendations = unseen_popular[:K].tolist()

    recalls.append(
        recall_at_k(recommendations, relevant_movies, K)
    )

    precisions.append(
        precision_at_k(recommendations, relevant_movies, K)
    )

    ndcgs.append(
        ndcg_at_k(recommendations, relevant_movies, K)
    )

print(f"Users evaluated: {len(recalls):,}")
print(f"Popularity Recall@{K}:    {np.mean(recalls):.4f}")
print(f"Popularity Precision@{K}: {np.mean(precisions):.4f}")
print(f"Popularity NDCG@{K}:      {np.mean(ndcgs):.4f}")

Users evaluated: 193,622
Popularity Recall@10:    0.0480
Popularity Precision@10: 0.0382
Popularity NDCG@10:      0.0530


In [74]:
# Keep only positive interactions for collaborative filtering

train_positive = train[train["positive"] == 1].copy()

print("Positive training interactions:", len(train_positive))
print("Unique users:", train_positive["userId"].nunique())
print("Unique movies:", train_positive["movieId"].nunique())

Positive training interactions: 11512202
Unique users: 200501
Unique movies: 43417


In [75]:
from scipy.sparse import csr_matrix

# Create integer IDs for rows and columns
user_ids = train_positive["userId"].unique()
movie_ids = train_positive["movieId"].unique()

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(movie_ids)
}

# Map each interaction to matrix coordinates
rows = train_positive["userId"].map(user_to_idx).values
cols = train_positive["movieId"].map(movie_to_idx).values

# Every positive interaction gets a value of 1
data = np.ones(len(train_positive), dtype=np.float32)

# Build sparse user × movie matrix
user_movie_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(user_ids), len(movie_ids))
)

print("Matrix shape:", user_movie_matrix.shape)
print("Non-zero interactions:", user_movie_matrix.nnz)

Matrix shape: (200501, 43417)
Non-zero interactions: 11512202


In [76]:
# Transpose the matrix so each row represents a movie
movie_user_matrix = user_movie_matrix.T.tocsr()

print("Movie-user matrix shape:", movie_user_matrix.shape)
print("Non-zero interactions:", movie_user_matrix.nnz)

Movie-user matrix shape: (43417, 200501)
Non-zero interactions: 11512202


In [77]:
from sklearn.neighbors import NearestNeighbors

# Number of neighbors to retrieve
N_NEIGHBORS = 51

item_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=N_NEIGHBORS,
    n_jobs=-1
)

item_model.fit(movie_user_matrix)

print("Item-KNN model fitted.")

Item-KNN model fitted.


In [78]:
# Pick a movie to inspect
movie_id = 318  # The Shawshank Redemption

# Find its index in our movie matrix
movie_idx = movie_to_idx[movie_id]

# Find its 10 nearest neighbors
distances, indices = item_model.kneighbors(
    movie_user_matrix[movie_idx],
    n_neighbors=11
)

# Convert cosine distance to cosine similarity
similarities = 1 - distances[0]

# Convert matrix indices back to movie IDs
similar_movie_ids = [
    movie_ids[idx]
    for idx in indices[0]
]

# Display the results
similar_movies = (
    pd.DataFrame({
        "movieId": similar_movie_ids,
        "similarity": similarities
    })
    .merge(movies, on="movieId", how="left")
)

display(similar_movies)

,movieId,similarity,title,genres
0,318,1.000000,"Shawshank Redemption, The (1994)",Crime|Drama
1,296,0.554534,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
2,593,0.546666,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
3,356,0.535930,Forrest Gump (1994),Comedy|Drama|Romance|War
4,527,0.535099,Schindler's List (1993),Drama|War
5,50,0.519693,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
6,2571,0.496506,"Matrix, The (1999)",Action|Sci-Fi|Thriller
7,2959,0.492586,Fight Club (1999),Action|Crime|Drama|Thriller
8,47,0.460503,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
9,858,0.453567,"Godfather, The (1972)",Crime|Drama


In [79]:
# Precompute the nearest neighbors for every movie

all_distances, all_indices = item_model.kneighbors(
    movie_user_matrix,
    n_neighbors=51
)

all_similarities = 1 - all_distances

print("Distances shape:", all_distances.shape)
print("Indices shape:", all_indices.shape)
print("Similarities shape:", all_similarities.shape)

Distances shape: (43417, 51)
Indices shape: (43417, 51)
Similarities shape: (43417, 51)


In [80]:
def recommend_item_knn(user_id, k=10):
    """
    Generate personalized recommendations for a user
    using item-item collaborative filtering.
    """

    # Get the user's positive training history
    user_movies = train_positive[
        train_positive["userId"] == user_id
    ]["movieId"].tolist()

    # Movies the user has already seen
    seen_movies = set(user_movies)

    # Store accumulated similarity scores
    candidate_scores = {}

    # Look at similar movies for every movie in the user's history
    for movie_id in user_movies:

        # Skip movies not present in our KNN matrix
        if movie_id not in movie_to_idx:
            continue

        movie_idx = movie_to_idx[movie_id]

        # Get precomputed neighbors and similarities
        neighbor_indices = all_indices[movie_idx]
        neighbor_similarities = all_similarities[movie_idx]

        for neighbor_idx, similarity in zip(
            neighbor_indices,
            neighbor_similarities
        ):
            candidate_movie_id = movie_ids[neighbor_idx]

            # Don't recommend movies already seen
            if candidate_movie_id in seen_movies:
                continue

            # Accumulate similarity score
            candidate_scores[candidate_movie_id] = (
                candidate_scores.get(candidate_movie_id, 0)
                + similarity
            )

    # Rank candidates by accumulated similarity
    ranked_movies = sorted(
        candidate_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Return top-K movie IDs
    recommendations = [
        movie_id
        for movie_id, score in ranked_movies[:k]
    ]

    return recommendations

In [81]:
user_id = 1

recommendations = recommend_item_knn(user_id, k=10)

recommendation_df = (
    pd.DataFrame({"movieId": recommendations})
    .merge(movies, on="movieId", how="left")
)

display(recommendation_df)

,movieId,title,genres
0,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure
1,1214,Alien (1979),Horror|Sci-Fi
2,608,Fargo (1996),Comedy|Crime|Drama|Thriller
3,1240,"Terminator, The (1984)",Action|Sci-Fi|Thriller
4,1136,Monty Python and the Holy Grail (1975),Adventure|Comedy|Fantasy
5,1265,Groundhog Day (1993),Comedy|Fantasy|Romance
6,1387,Jaws (1975),Action|Horror
7,1617,L.A. Confidential (1997),Crime|Film-Noir|Mystery|Thriller
8,2858,American Beauty (1999),Drama|Romance
9,924,2001: A Space Odyssey (1968),Adventure|Drama|Sci-Fi


In [82]:
import time

# Test recommendation speed on 1,000 users
sample_users = list(test_positive_by_user.index[:1000])

start = time.time()

for user_id in sample_users:
    recommend_item_knn(user_id, k=10)

elapsed = time.time() - start

print(f"Users tested: {len(sample_users):,}")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Average time/user: {elapsed / len(sample_users) * 1000:.2f} ms")

Users tested: 1,000
Total time: 3.26 seconds
Average time/user: 3.26 ms


In [83]:
# Precompute each user's positive training movie history

positive_movies_by_user = (
    train_positive
    .groupby("userId")["movieId"]
    .apply(list)
)

print("Users with positive history:", len(positive_movies_by_user))

# Check User 1
print("User 1 positive movies:", len(positive_movies_by_user.loc[1]))

Users with positive history: 200501
User 1 positive movies: 54


In [84]:
def recommend_item_knn(user_id, k=10):
    """
    Generate personalized recommendations for a user
    using item-item collaborative filtering.
    """

    # Get the user's positive training history directly
    user_movies = positive_movies_by_user.get(user_id, [])

    # Movies the user has already seen
    seen_movies = set(user_movies)

    # Store accumulated similarity scores
    candidate_scores = {}

    # Find similar movies for every movie in the user's history
    for movie_id in user_movies:

        # Skip movies not present in the KNN model
        if movie_id not in movie_to_idx:
            continue

        movie_idx = movie_to_idx[movie_id]

        # Retrieve precomputed neighbors and similarities
        neighbor_indices = all_indices[movie_idx]
        neighbor_similarities = all_similarities[movie_idx]

        for neighbor_idx, similarity in zip(
            neighbor_indices,
            neighbor_similarities
        ):
            candidate_movie_id = movie_ids[neighbor_idx]

            # Don't recommend movies already seen
            if candidate_movie_id in seen_movies:
                continue

            # Accumulate similarity scores
            candidate_scores[candidate_movie_id] = (
                candidate_scores.get(candidate_movie_id, 0)
                + similarity
            )

    # Rank candidates by score
    ranked_movies = sorted(
        candidate_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Return top-K movie IDs
    return [
        movie_id
        for movie_id, score in ranked_movies[:k]
    ]

In [85]:
import time

sample_users = list(test_positive_by_user.index[:1000])

start = time.time()

for user_id in sample_users:
    recommend_item_knn(user_id, k=10)

elapsed = time.time() - start

print(f"Users tested: {len(sample_users):,}")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Average time/user: {elapsed / len(sample_users) * 1000:.2f} ms")

Users tested: 1,000
Total time: 0.60 seconds
Average time/user: 0.60 ms


In [86]:
K = 10

recalls = []
precisions = []
ndcgs = []

start = time.time()

for user_id, relevant_movies in test_positive_by_user.items():

    recommendations = recommend_item_knn(
        user_id,
        k=K
    )

    recalls.append(
        recall_at_k(recommendations, relevant_movies, K)
    )

    precisions.append(
        precision_at_k(recommendations, relevant_movies, K)
    )

    ndcgs.append(
        ndcg_at_k(recommendations, relevant_movies, K)
    )

elapsed = time.time() - start

print(f"Users evaluated: {len(recalls):,}")
print(f"Item-KNN Recall@{K}:    {np.mean(recalls):.4f}")
print(f"Item-KNN Precision@{K}: {np.mean(precisions):.4f}")
print(f"Item-KNN NDCG@{K}:      {np.mean(ndcgs):.4f}")
print(f"Evaluation time: {elapsed:.2f} seconds")

Users evaluated: 193,622
Item-KNN Recall@10:    0.0644
Item-KNN Precision@10: 0.0456
Item-KNN NDCG@10:      0.0644
Evaluation time: 103.54 seconds


In [87]:
print("Recall statistics:")
print(pd.Series(recalls).describe())

print("\nNDCG statistics:")
print(pd.Series(ndcgs).describe())

print("\nFirst 20 Recall values:")
print(recalls[:20])

print("\nFirst 20 NDCG values:")
print(ndcgs[:20])

Recall statistics:
count    193622.000000
mean          0.064442
std           0.150620
min           0.000000
25%           0.000000
50%           0.000000
75%           0.052632
max           1.000000
dtype: float64

NDCG statistics:
count    193622.000000
mean          0.064360
std           0.128852
min           0.000000
25%           0.000000
50%           0.000000
75%           0.084314
max           1.000000
dtype: float64

First 20 Recall values:
[0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1111111111111111, 0.0, 0.0, 0.09090909090909091]

First 20 NDCG values:
[np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.21840743681816419), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.11752277470724268), np.float64(0.0), np.float64(0.0), np.float64(0.2200917662980801

In [88]:
# Integer mappings for BPR

bpr_user_ids = np.array(list(positive_movies_by_user.index))
bpr_movie_ids = np.array(movie_ids)

bpr_user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(bpr_user_ids)
}

bpr_movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(bpr_movie_ids)
}

print("BPR users:", len(bpr_user_ids))
print("BPR movies:", len(bpr_movie_ids))

BPR users: 200501
BPR movies: 43417


In [89]:
import random

# Number of training samples
N_SAMPLES = len(train_positive)

# Fast lookup of each user's positive movies
positive_sets = {
    user_id: set(movie_list)
    for user_id, movie_list in positive_movies_by_user.items()
}

# Random number generator for reproducibility
rng = np.random.default_rng(42)

bpr_users = np.empty(N_SAMPLES, dtype=np.int32)
bpr_positive = np.empty(N_SAMPLES, dtype=np.int32)
bpr_negative = np.empty(N_SAMPLES, dtype=np.int32)

for i, (user_id, positive_movie) in enumerate(
    train_positive[["userId", "movieId"]].itertuples(index=False, name=None)
):

    # Randomly sample an unseen movie
    while True:
        negative_movie = rng.choice(bpr_movie_ids)

        if negative_movie not in positive_sets[user_id]:
            break

    bpr_users[i] = bpr_user_to_idx[user_id]
    bpr_positive[i] = bpr_movie_to_idx[positive_movie]
    bpr_negative[i] = bpr_movie_to_idx[negative_movie]

print("BPR samples:", N_SAMPLES)
print("Users:", bpr_users.shape)
print("Positive movies:", bpr_positive.shape)
print("Negative movies:", bpr_negative.shape)

BPR samples: 11512202
Users: (11512202,)
Positive movies: (11512202,)
Negative movies: (11512202,)


In [90]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [91]:
import torch

print("PyTorch version:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

PyTorch version: 2.13.0
Device: CPU


In [92]:
import torch
import torch.nn as nn

# Number of latent factors
EMBEDDING_DIM = 64


class BPRMatrixFactorization(nn.Module):

    def __init__(self, n_users, n_movies, embedding_dim=64):
        super().__init__()

        self.user_embedding = nn.Embedding(
            n_users,
            embedding_dim
        )

        self.movie_embedding = nn.Embedding(
            n_movies,
            embedding_dim
        )

        # Learnable bias for each movie
        self.movie_bias = nn.Embedding(
            n_movies,
            1
        )

        # Initialize embeddings with small random values
        nn.init.normal_(
            self.user_embedding.weight,
            std=0.01
        )

        nn.init.normal_(
            self.movie_embedding.weight,
            std=0.01
        )

        nn.init.zeros_(
            self.movie_bias.weight
        )

    def forward(self, users, movies):

        user_vectors = self.user_embedding(users)
        movie_vectors = self.movie_embedding(movies)

        # Dot product between user and movie embeddings
        scores = (
            user_vectors * movie_vectors
        ).sum(dim=1)

        # Add movie bias
        scores += self.movie_bias(movies).squeeze()

        return scores


model = BPRMatrixFactorization(
    n_users=len(bpr_user_ids),
    n_movies=len(bpr_movie_ids),
    embedding_dim=EMBEDDING_DIM
)

print(model)

BPRMatrixFactorization(
  (user_embedding): Embedding(200501, 64)
  (movie_embedding): Embedding(43417, 64)
  (movie_bias): Embedding(43417, 1)
)


In [93]:
def bpr_loss(positive_scores, negative_scores):
    """
    Bayesian Personalized Ranking loss.

    Encourages the model to score positive movies
    higher than negative movies.
    """

    score_difference = positive_scores - negative_scores

    loss = -torch.mean(
        torch.log(
            torch.sigmoid(score_difference) + 1e-8
        )
    )

    return loss

In [94]:
from torch.utils.data import Dataset, DataLoader


class BPRDataset(Dataset):

    def __init__(self, users, positive_movies, negative_movies):
        self.users = torch.tensor(users, dtype=torch.long)
        self.positive_movies = torch.tensor(
            positive_movies,
            dtype=torch.long
        )
        self.negative_movies = torch.tensor(
            negative_movies,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return (
            self.users[idx],
            self.positive_movies[idx],
            self.negative_movies[idx]
        )


bpr_dataset = BPRDataset(
    bpr_users,
    bpr_positive,
    bpr_negative
)

print("Dataset size:", len(bpr_dataset))

# Inspect one training example
user, positive, negative = bpr_dataset[0]

print("User:", user.item())
print("Positive movie:", positive.item())
print("Negative movie:", negative.item())

Dataset size: 11512202
User: 25007
Positive movie: 0
Negative movie: 3875


In [95]:
BATCH_SIZE = 4096

bpr_loader = DataLoader(
    bpr_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

print("Number of batches:", len(bpr_loader))
print("Batch size:", BATCH_SIZE)

Number of batches: 2811
Batch size: 4096


In [96]:
# Check which hardware PyTorch can use

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Training device:", device)

Training device: mps


In [97]:
# Move the model to Apple's GPU
model = model.to(device)

# Adam optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print("Model device:", next(model.parameters()).device)
print("Optimizer:", optimizer.__class__.__name__)

Model device: mps:0
Optimizer: Adam


In [98]:
model.train()

# Take just one batch
users, positive_movies, negative_movies = next(iter(bpr_loader))

# Move batch to MPS
users = users.to(device)
positive_movies = positive_movies.to(device)
negative_movies = negative_movies.to(device)

# Forward pass
positive_scores = model(users, positive_movies)
negative_scores = model(users, negative_movies)

# Calculate BPR loss
loss = bpr_loss(
    positive_scores,
    negative_scores
)

# Backpropagation
optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Positive scores shape:", positive_scores.shape)
print("Negative scores shape:", negative_scores.shape)
print("BPR loss:", loss.item())

Positive scores shape: torch.Size([4096])
Negative scores shape: torch.Size([4096])
BPR loss: 0.6931511759757996


In [99]:
# Reset the BPR model so training starts from a fresh initialization

model = BPRMatrixFactorization(
    n_users=len(bpr_user_ids),
    n_movies=len(bpr_movie_ids),
    embedding_dim=EMBEDDING_DIM
).to(device)

# Reset optimizer for the new model
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print("BPR model reset.")
print("Device:", next(model.parameters()).device)

BPR model reset.
Device: mps:0


In [100]:
import time

model.train()

epoch_loss = 0.0
start_time = time.time()

for batch_idx, (users, positive_movies, negative_movies) in enumerate(bpr_loader):

    # Move batch to MPS
    users = users.to(device)
    positive_movies = positive_movies.to(device)
    negative_movies = negative_movies.to(device)

    # Calculate scores
    positive_scores = model(users, positive_movies)
    negative_scores = model(users, negative_movies)

    # BPR loss
    loss = bpr_loss(
        positive_scores,
        negative_scores
    )

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()

# Average loss across all batches
average_loss = epoch_loss / len(bpr_loader)

elapsed = time.time() - start_time

print(f"Epoch 1 completed")
print(f"Average BPR loss: {average_loss:.4f}")
print(f"Time: {elapsed / 60:.2f} minutes")

Epoch 1 completed
Average BPR loss: 0.2871
Time: 6.25 minutes


In [101]:
model.train()

epoch_loss = 0.0
start_time = time.time()

for batch_idx, (users, positive_movies, negative_movies) in enumerate(bpr_loader):

    users = users.to(device)
    positive_movies = positive_movies.to(device)
    negative_movies = negative_movies.to(device)

    positive_scores = model(users, positive_movies)
    negative_scores = model(users, negative_movies)

    loss = bpr_loss(
        positive_scores,
        negative_scores
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()

average_loss = epoch_loss / len(bpr_loader)
elapsed = time.time() - start_time

print("Epoch 2 completed")
print(f"Average BPR loss: {average_loss:.4f}")
print(f"Time: {elapsed / 60:.2f} minutes")

Epoch 2 completed
Average BPR loss: 0.1856
Time: 6.19 minutes


In [102]:
model.train()

epoch_loss = 0.0
start_time = time.time()

for batch_idx, (users, positive_movies, negative_movies) in enumerate(bpr_loader):

    users = users.to(device)
    positive_movies = positive_movies.to(device)
    negative_movies = negative_movies.to(device)

    positive_scores = model(users, positive_movies)
    negative_scores = model(users, negative_movies)

    loss = bpr_loss(
        positive_scores,
        negative_scores
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()

average_loss = epoch_loss / len(bpr_loader)
elapsed = time.time() - start_time

print("Epoch 3 completed")
print(f"Average BPR loss: {average_loss:.4f}")
print(f"Time: {elapsed / 60:.2f} minutes")

Epoch 3 completed
Average BPR loss: 0.1537
Time: 6.59 minutes


In [103]:
model.eval()

print("Model set to evaluation mode.")
print("User embedding shape:", model.user_embedding.weight.shape)
print("Movie embedding shape:", model.movie_embedding.weight.shape)

Model set to evaluation mode.
User embedding shape: torch.Size([200501, 64])
Movie embedding shape: torch.Size([43417, 64])


In [104]:
def recommend_bpr(user_id, k=10):
    """
    Generate Top-K personalized recommendations
    using the trained BPR matrix factorization model.
    """

    # Convert original userId → model index
    if user_id not in bpr_user_to_idx:
        return []

    user_idx = bpr_user_to_idx[user_id]

    # Get the user's learned embedding
    user_vector = model.user_embedding.weight[user_idx]

    # Get all movie embeddings
    movie_vectors = model.movie_embedding.weight

    # Calculate preference score for every movie
    scores = torch.matmul(
        movie_vectors,
        user_vector
    )

    # Add movie biases
    scores = scores + model.movie_bias.weight.squeeze()

    # Convert to CPU for processing
    scores = scores.detach().cpu().numpy()

    # Movies already seen during training
    seen_movies = train_seen_by_user.loc[user_id]

    # Convert seen movie IDs → model indices
    seen_indices = [
        bpr_movie_to_idx[movie_id]
        for movie_id in seen_movies
        if movie_id in bpr_movie_to_idx
    ]

    # Don't recommend previously seen movies
    scores[seen_indices] = -np.inf

    # Get the highest-scoring K movies
    top_indices = np.argsort(scores)[-k:][::-1]

    # Convert model indices → original MovieLens movie IDs
    recommendations = [
        bpr_movie_ids[idx]
        for idx in top_indices
    ]

    return recommendations

In [105]:
user_id = 1

bpr_recommendations = recommend_bpr(
    user_id,
    k=10
)

bpr_recommendation_df = (
    pd.DataFrame({
        "movieId": bpr_recommendations
    })
    .merge(
        movies,
        on="movieId",
        how="left"
    )
)

display(bpr_recommendation_df)

,movieId,title,genres
0,318,"Shawshank Redemption, The (1994)",Crime|Drama
1,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
2,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller
3,2959,Fight Club (1999),Action|Crime|Drama|Thriller
4,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
5,858,"Godfather, The (1972)",Crime|Drama
6,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure
7,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
8,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy
9,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy


In [106]:
# NDCG sanity check with a hand-computable example

recommended = [101, 200, 102, 300, 400, 500, 103, 600, 700, 800]
relevant = {101, 102, 103, 999}

recall = recall_at_k(recommended, relevant, k=10)
ndcg = ndcg_at_k(recommended, relevant, k=10)

print("Recommended:", recommended)
print("Relevant:", relevant)
print(f"Recall@10: {recall:.6f}")
print(f"NDCG@10:   {ndcg:.6f}")

Recommended: [101, 200, 102, 300, 400, 500, 103, 600, 700, 800]
Relevant: {103, 101, 102, 999}
Recall@10: 0.750000
NDCG@10:   0.715697


In [107]:
# Lock in the Item-KNN recall list under an unambiguous name before we add BPR's
itemknn_recalls = recalls

K = 10
bpr_recalls = []
bpr_precisions = []
bpr_ndcgs = []

for user_id, relevant_movies in test_positive_by_user.items():
    recommendations = recommend_bpr(user_id, k=K)
    bpr_recalls.append(recall_at_k(recommendations, relevant_movies, K))
    bpr_precisions.append(precision_at_k(recommendations, relevant_movies, K))
    bpr_ndcgs.append(ndcg_at_k(recommendations, relevant_movies, K))

print(f"BPR Recall@{K}:    {np.mean(bpr_recalls):.4f}")
print(f"BPR Precision@{K}: {np.mean(bpr_precisions):.4f}")
print(f"BPR NDCG@{K}:      {np.mean(bpr_ndcgs):.4f}")

BPR Recall@10:    0.0475
BPR Precision@10: 0.0381
BPR NDCG@10:      0.0527


In [108]:
for name in ["recalls", "ndcgs", "precisions",
             "test_positive_by_user", "train_seen_by_user",
             "recommend_bpr", "recommend_item_knn",
             "model", "train_positive", "all_indices",
             "bpr_movie_to_idx", "bpr_user_to_idx"]:
    print(f"{name:25s} {'OK' if name in dir() else 'MISSING'}")

recalls                   OK
ndcgs                     OK
precisions                OK
test_positive_by_user     OK
train_seen_by_user        OK
recommend_bpr             OK
recommend_item_knn        OK
model                     OK
train_positive            OK
all_indices               OK
bpr_movie_to_idx          OK
bpr_user_to_idx           OK


In [109]:
import numpy as np

# 1. Do different users get different recommendations?
sample_uids = list(test_positive_by_user.index[:5])
recs = {u: recommend_bpr(u, k=10) for u in sample_uids}
for u, r in recs.items():
    print(u, r)

first = set(recs[sample_uids[0]])
overlaps = [len(first & set(recs[u])) for u in sample_uids[1:]]
print("\nOverlap with first user's top-10:", overlaps)

# 2. How much does bias dominate the score?
with torch.no_grad():
    u_norm = model.user_embedding.weight.norm(dim=1).mean().item()
    m_norm = model.movie_embedding.weight.norm(dim=1).mean().item()
    bias_std = model.movie_bias.weight.std().item()
print(f"\nMean |user emb|: {u_norm:.4f}")
print(f"Mean |movie emb|: {m_norm:.4f}")
print(f"Movie bias std:  {bias_std:.4f}")
print(f"Typical dot-product magnitude: ~{u_norm * m_norm:.4f}")

1 [np.int64(318), np.int64(296), np.int64(2571), np.int64(2959), np.int64(4993), np.int64(858), np.int64(1198), np.int64(50), np.int64(5952), np.int64(7153)]
2 [np.int64(2571), np.int64(593), np.int64(2959), np.int64(260), np.int64(1196), np.int64(527), np.int64(4993), np.int64(858), np.int64(1198), np.int64(50)]
3 [np.int64(296), np.int64(2571), np.int64(2959), np.int64(4993), np.int64(50), np.int64(5952), np.int64(7153), np.int64(58559), np.int64(47), np.int64(79132)]
4 [np.int64(318), np.int64(296), np.int64(2571), np.int64(356), np.int64(593), np.int64(2959), np.int64(260), np.int64(1196), np.int64(527), np.int64(4993)]
5 [np.int64(2571), np.int64(356), np.int64(2959), np.int64(260), np.int64(1196), np.int64(527), np.int64(4993), np.int64(858), np.int64(1198), np.int64(50)]

Overlap with first user's top-10: [6, 7, 5, 6]

Mean |user emb|: 0.0000
Mean |movie emb|: 0.0000
Movie bias std:  0.5370
Typical dot-product magnitude: ~0.0000


In [112]:
import time, copy

model = BPRMatrixFactorization(
    n_users=len(bpr_user_ids),
    n_movies=len(bpr_movie_ids),
    embedding_dim=EMBEDDING_DIM,
).to(device)

with torch.no_grad():
    nn.init.normal_(model.user_embedding.weight, std=0.1)
    nn.init.normal_(model.movie_embedding.weight, std=0.1)
    nn.init.zeros_(model.movie_bias.weight)

# Regularization back on — this is what was missing
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=0.0)

best_val, best_state, best_epoch = -1, None, -1

for epoch in range(1, 6):
    model.train()
    epoch_loss, t0 = 0.0, time.time()

    for users, pos, neg in bpr_loader:
        users, pos, neg = users.to(device), pos.to(device), neg.to(device)
        loss = bpr_loss(model(users, pos), model(users, neg))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    model.eval()
    with torch.no_grad():
        u_norm = model.user_embedding.weight.norm(dim=1).mean().item()
        m_norm = model.movie_embedding.weight.norm(dim=1).mean().item()
    val_r, val_n = eval_bpr_sample(val_sample_users, val_positive_by_user)

    print(f"Epoch {epoch} | loss {epoch_loss/len(bpr_loader):.4f} | "
          f"val R@10 {val_r:.4f} | val NDCG@10 {val_n:.4f} | "
          f"|u| {u_norm:.4f} |i| {m_norm:.4f} | {(time.time()-t0)/60:.1f}m")

    # Keep only the BEST epoch, not the last
    if val_r > best_val:
        best_val, best_epoch = val_r, epoch
        best_state = copy.deepcopy(model.state_dict())
        torch.save({"model_state_dict": best_state, "epoch": epoch,
                    "val_recall": val_r}, "../models/bpr_model.pt")
        print(f"  ↳ new best (epoch {epoch}), checkpoint saved")

    # Stop if validation has fallen for two straight epochs
    if epoch - best_epoch >= 2:
        print(f"Early stopping. Best epoch: {best_epoch} (val R@10 {best_val:.4f})")
        break

model.load_state_dict(best_state)
model.eval()
print(f"\nLoaded best model from epoch {best_epoch}, val R@10 {best_val:.4f}")

Epoch 1 | loss 0.1128 | val R@10 0.0672 | val NDCG@10 0.0726 | |u| 1.9953 |i| 2.2808 | 4.0m
  ↳ new best (epoch 1), checkpoint saved
Epoch 2 | loss 0.0221 | val R@10 0.0663 | val NDCG@10 0.0714 | |u| 2.5396 |i| 3.0430 | 4.4m
Epoch 3 | loss 0.0042 | val R@10 0.0659 | val NDCG@10 0.0708 | |u| 2.7946 |i| 3.4773 | 5.7m
Early stopping. Best epoch: 1 (val R@10 0.0672)

Loaded best model from epoch 1, val R@10 0.0672


In [113]:
K = 10
bpr_recalls, bpr_precisions, bpr_ndcgs = [], [], []

t0 = time.time()
for user_id, relevant_movies in test_positive_by_user.items():
    rec = recommend_bpr(user_id, k=K)
    bpr_recalls.append(recall_at_k(rec, relevant_movies, K))
    bpr_precisions.append(precision_at_k(rec, relevant_movies, K))
    bpr_ndcgs.append(ndcg_at_k(rec, relevant_movies, K))

print(f"Users evaluated:   {len(bpr_recalls):,}")
print(f"BPR Recall@{K}:    {np.mean(bpr_recalls):.4f}")
print(f"BPR Precision@{K}: {np.mean(bpr_precisions):.4f}")
print(f"BPR NDCG@{K}:      {np.mean(bpr_ndcgs):.4f}")
print(f"Time: {(time.time()-t0)/60:.1f} min")

Users evaluated:   193,622
BPR Recall@10:    0.0543
BPR Precision@10: 0.0421
BPR NDCG@10:      0.0584
Time: 10.1 min


In [114]:
itemknn_recalls = recalls   # only if not already saved under this name

from scipy.stats import wilcoxon
stat, p = wilcoxon(itemknn_recalls, bpr_recalls)
print(f"Wilcoxon p-value: {p:.6f}")
print(f"Item-KNN mean R@10: {np.mean(itemknn_recalls):.4f}")
print(f"BPR mean R@10:      {np.mean(bpr_recalls):.4f}")

Wilcoxon p-value: 0.000000
Item-KNN mean R@10: 0.0644
BPR mean R@10:      0.0543


In [115]:
import pandas as pd
pd.DataFrame({
    "model": ["Popularity", "BPR (MF)", "Item-KNN"],
    "recall@10": [0.0480, np.mean(bpr_recalls), np.mean(itemknn_recalls)],
    "precision@10": [0.0382, np.mean(bpr_precisions), np.mean(precisions)],
    "ndcg@10": [0.0530, np.mean(bpr_ndcgs), np.mean(ndcgs)],
}).to_csv("../data/processed/leaderboard.csv", index=False)

np.save("../data/processed/bpr_test_recalls.npy", np.array(bpr_recalls))
np.save("../data/processed/itemknn_test_recalls.npy", np.array(itemknn_recalls))
print("Saved leaderboard and per-user metrics.")

Saved leaderboard and per-user metrics.


In [116]:
import pandas as pd

eval_users = list(test_positive_by_user.index)

# --- History length per evaluated user (positive training interactions) ---
hist_len = {u: len(positive_movies_by_user.get(u, [])) for u in eval_users}

seg = pd.DataFrame({
    "userId": eval_users,
    "history": [hist_len[u] for u in eval_users],
    "itemknn": itemknn_recalls,
    "bpr": bpr_recalls,
})

seg["bucket"] = pd.cut(
    seg["history"],
    bins=[-1, 20, 50, 200, np.inf],
    labels=["1-20", "21-50", "51-200", "200+"],
)

print("===== RECALL@10 BY USER HISTORY LENGTH =====")
by_bucket = seg.groupby("bucket", observed=True).agg(
    users=("userId", "size"),
    itemknn_r10=("itemknn", "mean"),
    bpr_r10=("bpr", "mean"),
)
by_bucket["bpr_minus_knn"] = by_bucket["bpr_r10"] - by_bucket["itemknn_r10"]
print(by_bucket.round(4))

# --- Popularity bias: where do recommended items sit in the popularity ranking? ---
pop_rank = {mid: r for r, mid in enumerate(movie_positive_counts.index, start=1)}
n_items = len(pop_rank)

def mean_pop_rank(rec_fn, users, k=10):
    ranks = []
    for u in users:
        for mid in rec_fn(u, k=k):
            if mid in pop_rank:
                ranks.append(pop_rank[mid])
    return np.mean(ranks), np.median(ranks)

sample = eval_users[:2000]
knn_mean, knn_med = mean_pop_rank(recommend_item_knn, sample)
bpr_mean, bpr_med = mean_pop_rank(recommend_bpr, sample)

print(f"\n===== POPULARITY BIAS (n={len(sample)} users, {n_items:,} ranked items) =====")
print(f"Item-KNN  mean pop-rank: {knn_mean:8.1f}   median: {knn_med:8.1f}")
print(f"BPR       mean pop-rank: {bpr_mean:8.1f}   median: {bpr_med:8.1f}")
print("(Popularity baseline recommends ranks 1-10 by construction. "
      "Higher = digging deeper into the catalog.)")

===== RECALL@10 BY USER HISTORY LENGTH =====
        users  itemknn_r10  bpr_r10  bpr_minus_knn
bucket                                            
1-20    67058       0.0962   0.0767        -0.0195
21-50   60956       0.0619   0.0524        -0.0095
51-200  55983       0.0372   0.0354        -0.0018
200+     9625       0.0173   0.0202         0.0028

===== POPULARITY BIAS (n=2000 users, 43,417 ranked items) =====
Item-KNN  mean pop-rank:    133.2   median:     28.0
BPR       mean pop-rank:     59.9   median:     20.0
(Popularity baseline recommends ranks 1-10 by construction. Higher = digging deeper into the catalog.)


In [117]:
# Is the 200+ flip statistically real?
heavy = seg[seg["bucket"] == "200+"]
stat, p = wilcoxon(heavy["itemknn"], heavy["bpr"])
print(f"200+ bucket: n={len(heavy):,}, Item-KNN {heavy['itemknn'].mean():.4f}, "
      f"BPR {heavy['bpr'].mean():.4f}, p={p:.6f}")

# How much of the catalog does each model actually touch?
def coverage(rec_fn, users, k=10):
    seen = set()
    for u in users:
        seen.update(rec_fn(u, k=k))
    return len(seen)

print(f"\nCatalog coverage over {len(sample):,} users ({n_items:,} items):")
print(f"  Item-KNN: {coverage(recommend_item_knn, sample):,} distinct items")
print(f"  BPR:      {coverage(recommend_bpr, sample):,} distinct items")
print(f"  Popularity: 10 (by construction)")

200+ bucket: n=9,625, Item-KNN 0.0173, BPR 0.0202, p=0.000000

Catalog coverage over 2,000 users (43,417 items):
  Item-KNN: 698 distinct items
  BPR:      410 distinct items
  Popularity: 10 (by construction)


In [118]:
def recommend_item_knn_normalized(user_id, k=10):
    """Item-KNN with scores normalized by history length."""
    user_movies = positive_movies_by_user.get(user_id, [])
    seen_movies = set(user_movies)
    candidate_scores = {}

    n_hist = 0
    for movie_id in user_movies:
        if movie_id not in movie_to_idx:
            continue
        n_hist += 1
        movie_idx = movie_to_idx[movie_id]
        for neighbor_idx, similarity in zip(all_indices[movie_idx],
                                            all_similarities[movie_idx]):
            candidate_movie_id = movie_ids[neighbor_idx]
            if candidate_movie_id in seen_movies:
                continue
            candidate_scores[candidate_movie_id] = (
                candidate_scores.get(candidate_movie_id, 0) + similarity
            )

    if n_hist > 0:
        for mid in candidate_scores:
            candidate_scores[mid] /= n_hist

    return [mid for mid, _ in sorted(candidate_scores.items(),
                                     key=lambda x: x[1], reverse=True)[:k]]

In [119]:
K = 10
norm_recalls, norm_precisions, norm_ndcgs = [], [], []

t0 = time.time()
for user_id, relevant_movies in test_positive_by_user.items():
    rec = recommend_item_knn_normalized(user_id, k=K)
    norm_recalls.append(recall_at_k(rec, relevant_movies, K))
    norm_precisions.append(precision_at_k(rec, relevant_movies, K))
    norm_ndcgs.append(ndcg_at_k(rec, relevant_movies, K))

print(f"Item-KNN (normalized) Recall@{K}:    {np.mean(norm_recalls):.4f}")
print(f"Item-KNN (normalized) Precision@{K}: {np.mean(norm_precisions):.4f}")
print(f"Item-KNN (normalized) NDCG@{K}:      {np.mean(norm_ndcgs):.4f}")
print(f"(baseline Item-KNN Recall@10 was 0.0644)")
print(f"Time: {(time.time()-t0)/60:.1f} min")

Item-KNN (normalized) Recall@10:    0.0644
Item-KNN (normalized) Precision@10: 0.0456
Item-KNN (normalized) NDCG@10:      0.0644
(baseline Item-KNN Recall@10 was 0.0644)
Time: 1.8 min


In [120]:
def recommend_item_knn_capped(user_id, k=10, max_history=50, mode="recent"):
    """
    Item-KNN restricted to a subset of the user's history.
    mode="recent"  -> most recent max_history positive interactions
    mode="similar" -> the max_history items with the strongest neighbor signal
    """
    user_movies = positive_movies_by_user.get(user_id, [])
    seen_movies = set(user_movies)

    valid = [m for m in user_movies if m in movie_to_idx]

    if mode == "recent":
        # positive_movies_by_user preserves chronological order from the sorted train frame
        hist = valid[-max_history:]
    else:  # "similar": keep items whose own top neighbors are strongest
        scored = [(m, all_similarities[movie_to_idx[m]][1:].mean()) for m in valid]
        scored.sort(key=lambda x: x[1], reverse=True)
        hist = [m for m, _ in scored[:max_history]]

    candidate_scores = {}
    for movie_id in hist:
        movie_idx = movie_to_idx[movie_id]
        for neighbor_idx, similarity in zip(all_indices[movie_idx],
                                            all_similarities[movie_idx]):
            cid = movie_ids[neighbor_idx]
            if cid in seen_movies:
                continue
            candidate_scores[cid] = candidate_scores.get(cid, 0) + similarity

    return [mid for mid, _ in sorted(candidate_scores.items(),
                                     key=lambda x: x[1], reverse=True)[:k]]

In [121]:
K = 10
results = {}

for mode in ["recent", "similar"]:
    rs = []
    t0 = time.time()
    for user_id, relevant in test_positive_by_user.items():
        rec = recommend_item_knn_capped(user_id, k=K, max_history=50, mode=mode)
        rs.append(recall_at_k(rec, relevant, K))
    results[mode] = rs
    print(f"{mode:8s} overall R@10: {np.mean(rs):.4f}  ({(time.time()-t0)/60:.1f} min)")

# Segment breakdown vs the original
seg2 = seg.copy()
seg2["knn_recent"] = results["recent"]
seg2["knn_similar"] = results["similar"]

print("\n===== RECALL@10 BY HISTORY BUCKET =====")
print(seg2.groupby("bucket", observed=True).agg(
    users=("userId", "size"),
    knn_orig=("itemknn", "mean"),
    knn_recent=("knn_recent", "mean"),
    knn_similar=("knn_similar", "mean"),
    bpr=("bpr", "mean"),
).round(4))

recent   overall R@10: 0.0645  (1.0 min)
similar  overall R@10: 0.0641  (1.3 min)

===== RECALL@10 BY HISTORY BUCKET =====
        users  knn_orig  knn_recent  knn_similar     bpr
bucket                                                  
1-20    67058    0.0962      0.0962       0.0962  0.0767
21-50   60956    0.0619      0.0619       0.0619  0.0524
51-200  55983    0.0372      0.0374       0.0362  0.0354
200+     9625    0.0173      0.0165       0.0166  0.0202


In [122]:
# How many test positives does each bucket actually have?
seg2["n_relevant"] = [len(test_positive_by_user[u]) for u in seg2["userId"]]

print(seg2.groupby("bucket", observed=True).agg(
    users=("userId", "size"),
    mean_relevant=("n_relevant", "mean"),
    median_relevant=("n_relevant", "median"),
    knn_recall=("itemknn", "mean"),
    knn_precision_proxy=("itemknn", lambda s: np.nan),  # placeholder, see below
).round(2))

# Precision@10 isn't capped by target count — check whether the collapse persists
knn_prec_by_user = precisions  # per-user Item-KNN precision from your original run
seg2["knn_prec"] = knn_prec_by_user
print("\n===== PRECISION@10 BY BUCKET (not denominator-capped) =====")
print(seg2.groupby("bucket", observed=True).agg(
    knn_precision=("knn_prec", "mean"),
).round(4))

        users  mean_relevant  median_relevant  knn_recall  knn_precision_proxy
bucket                                                                        
1-20    67058           2.90              3.0        0.10                  NaN
21-50   60956           6.66              6.0        0.06                  NaN
51-200  55983          18.30             16.0        0.04                  NaN
200+     9625          59.34             51.0        0.02                  NaN

===== PRECISION@10 BY BUCKET (not denominator-capped) =====
        knn_precision
bucket               
1-20           0.0273
21-50          0.0411
51-200         0.0645
200+           0.0909


In [123]:
# Recall@K with K scaled to each user's target count — removes the denominator cap
def recall_at_n(rec_fn, users, truth):
    out = []
    for u in users:
        rel = truth[u]
        n = min(len(rel), 100)          # cap for runtime
        rec = rec_fn(u, k=n)
        out.append(len(set(rec) & rel) / len(rel))
    return np.mean(out)

sample = list(test_positive_by_user.index[:20000])
seg_s = seg2[seg2["userId"].isin(sample)]

print("Recall@R (K = each user's number of relevant items):")
for b in ["1-20", "21-50", "51-200", "200+"]:
    us = seg_s[seg_s["bucket"] == b]["userId"].tolist()
    if us:
        print(f"  {b:8s} {recall_at_n(recommend_item_knn, us, test_positive_by_user):.4f}")
        

Recall@R (K = each user's number of relevant items):
  1-20     0.0343
  21-50    0.0442
  51-200   0.0571
  200+     0.0702
